In [ ]:
!pip  install  -q  groq langgraph  langchain-groq duckduckgo-search


In [ ]:
import json
import re
from  typing  import  TypedDict , Annotated
from groq import Groq
from  google.colab  import userdata

from langgraph.graph  import  StateGraph, END
from langgraph.prebuilt import  ToolNode
from langchain_groq  import ChatGroq
from  langchain_core.messages import HumanMessage , AIMessage ,SystemMessage
from langchain_core.tools import tool
from duckduckgo_search import DDGS


GROQ_API_KEY = userdata.get("GROQ_API_KEY")
groq_client  = Groq(api_key=GROQ_API_KEY)


# qwen/qwen3.6-27b   — replaces llama-3.3-70b-versatile (deprecated Aug 16 2026)
# openai/gpt-oss-20b — replaces llama-3.1-8b-instant    (deprecated Aug 16 2026)
GROQ_MODEL      = "qwen/qwen3.6-27b"       # direct client calls
LANGGRAPH_MODEL = "openai/gpt-oss-20b"     # LangGraph / LangChain binding

llm = ChatGroq(api_key=GROQ_API_KEY, model=LANGGRAPH_MODEL)

print(f"Direct Groq client model : {GROQ_MODEL}")
print(f"LangGraph model          : {LANGGRAPH_MODEL}")
print("Setup complete.")

Direct Groq client model : qwen/qwen3.6-27b
LangGraph model          : openai/gpt-oss-20b
Setup complete.


Segment 1 — LLM + Tools: Function Calling

In [ ]:
#what is the weather in Delhi right now?
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    weather_data = {
        "london": "12°C, cloudy with light rain",
        "paris":  "18°C, partly sunny",
        "tokyo":  "24°C, clear skies",
        "mumbai": "31°C, humid and hazy",
    }
    return weather_data.get(city.lower(), f"Weather data unavailable for {city}")

In [ ]:
def calculator(expression: str) -> str:
    """Evaluate a mathematical arithmetic expression e.g. '847*0.15+234'."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def get_stock_price(ticker: str) -> str:
    """Get current stock price for a ticker symbol."""
    prices = {"AAPL": "$189.30", "GOOGL": "$174.50", "MSFT": "$415.20"}
    return prices.get(ticker.upper(), f"Price unavailable for {ticker}")


In [ ]:
TOOL_MAP = {
    "get_weather":     get_weather,
    "calculator":      calculator,
    "get_stock_price": get_stock_price,
}


In [ ]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical arithmetic expression. Pass expression as a single string with no spaces e.g. '847*0.15+234'",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Valid arithmetic expression e.g. '23*47'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get stock price for a ticker symbol",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string"}
                },
                "required": ["ticker"]
            }
        }
    },
]

print("Tools defined:")
for name in TOOL_MAP:
    print(f"  {name}")

Tools defined:
  get_weather
  calculator
  get_stock_price


In [ ]:
# ============================================================
# Single-turn tool call: LLM decides which tool, we execute it
# ============================================================

def run_with_tools(user_query: str, verbose: bool = True) -> str:
    messages = [{"role": "user", "content": user_query}]

    # Step 1: LLM decides if a tool is needed and which one
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=messages,
            tools=TOOLS_SCHEMA,
            tool_choice="auto"
        )
    except Exception as e:
        if verbose:
            print(f"  Tool call failed ({e.__class__.__name__}) — falling back to direct answer")
        fallback = groq_client.chat.completions.create(
            model=GROQ_MODEL, messages=messages
        )
        return fallback.choices[0].message.content

    msg = response.choices[0].message

    # Step 2: No tool needed — return direct answer
    if not msg.tool_calls:
        return msg.content

    # Step 3: Execute each tool the LLM requested
    messages.append({
        "role":       "assistant",
        "content":    msg.content or "",
        "tool_calls": msg.tool_calls
    })

    for tool_call in msg.tool_calls:
        fn_name = tool_call.function.name
        try:
            fn_args = json.loads(tool_call.function.arguments)
            result  = TOOL_MAP[fn_name](**fn_args)
        except Exception as e:
            result = f"Tool execution error: {e}"

        if verbose:
            print(f"  Tool called: {fn_name}({fn_args})")
            print(f"  Tool result: {result}")

        messages.append({
            "role":         "tool",
            "tool_call_id": tool_call.id,
            "content":      result
        })

    # Step 4: LLM generates final answer with tool results in context
    try:
        final = groq_client.chat.completions.create(
            model=GROQ_MODEL, messages=messages
        )
        return final.choices[0].message.content
    except Exception as e:
        return f"Error generating final answer: {e}"


test_queries = [
    "What's the weather like in Tokyo?",
    "What is 15% of 847 plus 234?",
    "What's the stock price of Apple?",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    answer = run_with_tools(q)
    print(f"Answer: {answer}")
    print("-" * 50)


Query: What's the weather like in Tokyo?
  Tool called: get_weather({'city': 'Tokyo'})
  Tool result: 24°C, clear skies
Answer: 
<think>
The user asked about the weather in Tokyo. I called a weather tool for Tokyo and got the result: 24°C, clear skies. I should provide this information to the user in a clear and concise manner. There's no complexity in the response, just deliver the facts. I will format it naturally.
</think>

Currently, the weather in Tokyo is 24°C with clear skies. Not too bad!
--------------------------------------------------

Query: What is 15% of 847 plus 234?
  Tool called: calculator({'expression': '847*0.15+234'})
  Tool result: 847*0.15+234 = 361.05
Answer: 
<think>
The user wants to calculate 15% of 847 and then add 234 to the result.
I have used the calculator tool and got the result 361.05.
I will formulate the answer based on this result.
Steps:
1. Calculate 15% of 847: $847 \times 0.15 = 127.05$
2. Add 234 to the result: $127.05 + 234 = 361.05$
The fina

In [ ]:
# ============================================================
# ReAct = Reason + Act
# Pattern: Thought → Action → Observation → repeat → Final Answer
# ============================================================


REACT_SYSTEM = """You are a helpful assistant that reasons step by step.
For each step, output in this exact format:

Thought: [your reasoning about what to do next]
Action: [tool_name]
Action Input: [arguments as JSON]

When you have enough information, output:
Thought: I now have enough information to answer.
Final Answer: [your complete answer]

Available tools:
- get_weather(city): get current weather for a city
- calculator(expression): evaluate math e.g. {"expression": "23*47"}
- get_stock_price(ticker): get stock price for a ticker

Important: Action Input must be valid JSON with no spaces around operators."""


def parse_react_step(text: str):
    if "Final Answer:" in text:
        return "final", text.split("Final Answer:")[-1].strip(), None

    action_match = re.search(r"Action:\s*(\w+)", text)
    input_match  = re.search(r"Action Input:\s*(\{.*?\})", text, re.DOTALL)

    if action_match and input_match:
        action = action_match.group(1).strip()
        try:
            args = json.loads(input_match.group(1).strip())
        except json.JSONDecodeError:
            return "unknown", text, None
        return "action", action, args

    if "final answer" in text.lower():
        return "final", text, None

    return "unknown", text, None


def react_agent(query: str, max_steps: int = 6):
    messages = [
        {"role": "system", "content": REACT_SYSTEM},
        {"role": "user",   "content": query}
    ]

    print(f"Query: {query}\n{'='*55}")
    recent_calls = []

    for step in range(max_steps):
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=messages,
                temperature=0
            )
        except Exception as e:
            print(f"LLM call failed: {e}")
            break

        llm_output = response.choices[0].message.content
        print(f"\n[Step {step+1}]\n{llm_output}")

        step_type, value, args = parse_react_step(llm_output)

        if step_type == "final":
            print(f"\n{'='*55}\nFinal Answer: {value}")
            return value

        if step_type == "action":
            call_sig = f"{value}:{json.dumps(args, sort_keys=True)}"
            if call_sig in recent_calls[-3:]:
                print("Loop detected — stopping.")
                break
            recent_calls.append(call_sig)

            if value not in TOOL_MAP:
                observation = f"Error: unknown tool '{value}'. Available: {list(TOOL_MAP.keys())}"
            else:
                try:
                    observation = TOOL_MAP[value](**args)
                except Exception as e:
                    observation = f"Tool error: {e}"

            print(f"\nObservation: {observation}")
            messages.append({"role": "assistant", "content": llm_output})
            messages.append({"role": "user",      "content": f"Observation: {observation}"})

        else:
            print("Could not parse output — nudging model.")
            messages.append({"role": "assistant", "content": llm_output})
            messages.append({
                "role": "user",
                "content": "Please follow the exact format: Thought / Action / Action Input or Final Answer."
            })

    return "Max steps reached without final answer."


react_agent("What's the weather in London and what is 23 multiplied by 47?")

Query: What's the weather in London and what is 23 multiplied by 47?

[Step 1]

<think>
The user is asking for two things:
1. The weather in London.
2. The result of 23 multiplied by 47.

I need to use two tools:
1. `get_weather` with the argument `city="London"`.
2. `calculator` with the argument `expression="23*47"`.

I can perform these actions in parallel since they are independent.

Step 1: Get weather for London.
Step 2: Calculate 23 * 47.
Step 3: Combine the results.

Let's start with the tool calls.
</think>

Thought: The user wants to know the weather in London and the result of a multiplication. I need to call the `get_weather` tool for London and the `calculator` tool for the expression "23*47".
Action: get_weather
Action Input: {"city": "London"}
Action: calculator
Action Input: {"expression": "23*47"}

Observation: 12°C, cloudy with light rain

[Step 2]

<think>
The user asked for the weather in London and the result of 23 multiplied by 47.
I have already called the `get_w

'The weather in London is 12°C, cloudy with light rain. 23 multiplied by 47 is 1081.'

## Segment 3 — LangGraph: Nodes, Edges, State

In [ ]:
@tool
def  web_search(query:str) -> str:
  """ Search the web for Current  information , recent events , an news"""
  try:
    with DDGS() as ddgs:
      results = list(ddgs.text(query,max_results=3))
    if not results:
      return  "No results Found"
    return "\n\n".join(
        f"Title:{r['title']}\nURL: {r['href']}\nSnippet:{r['body']}"
        for r in results
    )

  except Exception as e:
    return f"Search error:{e}"

In [ ]:
@tool
def math_tool(expression: str) -> str:
    """Evaluate a mathematical expression e.g. '23*47'."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"

In [ ]:
AGENT_TOOLS    = [web_search, math_tool]
llm_with_tools = llm.bind_tools(AGENT_TOOLS)

print(f"LangGraph tools bound to: {LANGGRAPH_MODEL}")
for t in AGENT_TOOLS:
    print(f"  {t.name}: {t.description}")

LangGraph tools bound to: openai/gpt-oss-20b
  web_search: Search the web for Current  information , recent events , an news
  math_tool: Evaluate a mathematical expression e.g. '23*47'.


In [ ]:
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def agent_node(state: AgentState) -> AgentState:
    """LLM decides: answer directly or call a tool."""
    system = SystemMessage(content="""You are a helpful research assistant.
Use web_search for current information and recent events.
Use math_tool for calculations.
Be concise in your final answers.""")

    messages = [system] + state["messages"]
    try:
        response = llm_with_tools.invoke(messages)
    except Exception as e:
        response = AIMessage(content=f"Error: {e}. Please try rephrasing.")
    return {"messages": [response]}


tool_node = ToolNode(AGENT_TOOLS)

def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

In [ ]:
graph_builder = StateGraph(AgentState)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", tool_node)
graph_builder.set_entry_point("agent")
graph_builder.add_conditional_edges(
    "agent", should_continue, {"tools": "tools", END: END}
)
graph_builder.add_edge("tools", "agent")

agent_graph = graph_builder.compile()

print("LangGraph compiled successfully.")
print("\nGraph structure:")
print("  START → agent → [tool call?] → tools → agent → ... → END")
print("                → [no tool]   → END")

LangGraph compiled successfully.

Graph structure:
  START → agent → [tool call?] → tools → agent → ... → END
                → [no tool]   → END


In [ ]:
# ============================================================
# In-context memory: messages accumulate in state across turns
# ============================================================

def run_agent(query: str, history: list = None, verbose: bool = True):
    if history is None:
        history = []
    history.append(HumanMessage(content=query))
    final_state = agent_graph.invoke({"messages": history})
    response    = final_state["messages"][-1].content
    if verbose:
        print(f"User:  {query}")
        print(f"Agent: {response}\n")
    return response, final_state["messages"]

In [ ]:
print("=== Multi-turn conversation (in-context memory) ===\n")
history = []
_, history = run_agent("My name is siddhesh and I am a ML Engineer.", history)

=== Multi-turn conversation (in-context memory) ===

User:  My name is siddhesh and I am a ML Engineer.
Agent: Nice to meet you, Siddhesh! If there’s anything related to ML or any other topic you’d like to dive into, just let me know.



In [ ]:
_, history = run_agent("What is 144 divided by 12?", history)

User:  What is 144 divided by 12?
Agent: 144 ÷ 12 = **12**.



In [ ]:
_, history = run_agent("Based on my role, what agent framework would you recommend I learn  first?", history)

User:  Based on my role, what agent framework would you recommend I learn  first?
Agent: **For an ML Engineer like you, the first agent framework to learn is **_LangChain_**.**

| Why LangChain? | Details |
|-----------------|---------|
| **LLM‑centric** | Built specifically to orchestrate large‑language‑model workflows (prompting, chaining, memory, retrieval). |
| **Python‑friendly** | Uses the same ecosystem you’re already comfortable with (NumPy, PyTorch, scikit‑learn). |
| **Extensible** | Plug‑in custom components (retrievers, tools, callbacks) without re‑writing the core. |
| **Community & Docs** | Active open‑source project, rich tutorials, and plenty of example agents (question‑answering, data‑analysis, multi‑step reasoning). |
| **Deployment** | Works with Docker, FastAPI, and cloud services (AWS Lambda, Azure Functions). |

**Next steps**

1. **Install**: `pip install langchain`
2. **Tutorial**: Follow the official LangChain quickstart – build a simple “retrieval‑augmented” c

In [ ]:
print(f"Conversation turns:        {len([m for m in history if isinstance(m, HumanMessage)])}")
print(f"Total messages in context: {len(history)}")

Conversation turns:        3
Total messages in context: 8


In [ ]:
# ============================================================
# External memory: persist and retrieve facts across sessions
# In production: replace dict with a vector DB (Chroma, FAISS)
# ============================================================

MEMORY_STORE = {}


def save_to_memory(key: str, value: str):
    MEMORY_STORE[key] = value
    print(f"  Saved: [{key}] = {value[:60]}")

def retrieve_from_memory(key: str) -> str:
    return MEMORY_STORE.get(key, "No memory found.")

def build_memory_context(keys: list) -> str:
    items = [f"{k}: {MEMORY_STORE[k]}" for k in keys if k in MEMORY_STORE]
    return "Relevant context:\n" + "\n".join(items) if items else ""

In [ ]:
save_to_memory("user_name", "Siddhesh Dhande")
save_to_memory("user_role", "Machine learning Engineer")
save_to_memory("topics",    "Neural networks, LLMs, RAG, Agents")

  Saved: [user_name] = Siddhesh Dhande
  Saved: [user_role] = Machine learning Engineer
  Saved: [topics] = Neural networks, LLMs, RAG, Agents


In [ ]:
# Session 2 — new context, inject memory into prompt
memory_ctx = build_memory_context(["user_name", "user_role"])
query_with_memory = f"{memory_ctx}\n\nUser: Recommend one beginner-friendly project for building LangGraph."

In [ ]:
print("\n=== External memory injected into new session ===\n")
# Capture the return values, but don't explicitly print them again,
# as run_agent already prints the user query and agent response.
_response, _history = run_agent(query_with_memory, history=None)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


=== External memory injected into new session ===



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

User:  Relevant context:
user_name: Siddhesh Dhande
user_role: Machine learning Engineer

User: Recommend one beginner-friendly project for building LangGraph.
Agent: **Beginner‑Friendly LangGraph Project: “Smart Recipe Recommender”**

---

### 1. Goal  
Create a conversational agent that:

1. **Accepts user preferences** (e.g., dietary restrictions, ingredients on hand, meal type).
2. **Retrieves relevant recipes** from a local vector store (FAISS) or a simple JSON list.
3. **Generates a step‑by‑step cooking guide** using an LLM.
4. **Keeps context** across turns (e.g., remembers user’s preferred cuisine).

---

### 2. Tech Stack  

| Component | Why it fits |
|-----------|-------------|
| **LangGraph** | Handles stateful flow, node orchestration, and LLM calls. |
| **OpenAI GPT‑4o / Anthropic Claude** | LLM for natural language generation. |
| **FAISS** | Lightweight vector search for recipe embeddings. |
| **Python 3.10+** | Standard for ML engineers. |
| **FastAPI** (optional) | Ex

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
print("\nMemory type summary:")
for name, desc in {
    "In-context":  "Messages in current window — fast, limited by context length",
    "External":    "Vector DB / dict — persists across sessions, retrieved by key or similarity",
    "Episodic":    "Compressed past-session summaries — reduces token cost on long histories",
    "Semantic":    "Structured knowledge base — facts the agent can query explicitly",
}.items():
    print(f"  {name:<12} {desc}")


Memory type summary:
  In-context   Messages in current window — fast, limited by context length
  External     Vector DB / dict — persists across sessions, retrieved by key or similarity
  Episodic     Compressed past-session summaries — reduces token cost on long histories
  Semantic     Structured knowledge base — facts the agent can query explicitly


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ============================================================
# Common agent failure modes — demonstrated with fixes
# ============================================================

print("=== Failure Mode 1: Hallucinated tool calls ===")

def safe_tool_execute(tool_name: str, args: dict) -> str:
    if tool_name not in TOOL_MAP:
        return f"[ERROR] Tool '{tool_name}' does not exist. Available: {list(TOOL_MAP.keys())}"
    return TOOL_MAP[tool_name](**args)

print(safe_tool_execute("nonexistent_tool", {"query": "test"}))
print(safe_tool_execute("calculator", {"expression": "12*8"}))


print("\n=== Failure Mode 2: Infinite loops ===")

def loop_detector(call_history: list, current_call: dict) -> bool:
    return current_call in call_history[-3:]

call_history = [
    {"tool": "web_search", "args": {"query": "latest AI news"}},
    {"tool": "web_search", "args": {"query": "latest AI news"}},
]
new_call = {"tool": "web_search", "args": {"query": "latest AI news"}}
is_loop = loop_detector(call_history, new_call)
print(f"Loop detected: {is_loop} → {'stop and return partial answer' if is_loop else 'continue'}")


print("\n=== Failure Mode 3: Context window overflow ===")

def truncate_tool_output(output: str, max_chars: int = 500) -> str:
    if len(output) <= max_chars:
        return output
    return output[:max_chars] + f"\n[Truncated — {len(output)-max_chars} chars omitted]"

long_output = "A" * 2000
print(f"Original length:  {len(long_output)}")
print(f"Truncated length: {len(truncate_tool_output(long_output))}")


print("\n=== Failure mode checklist ===")
FAILURE_MODES = {
    "Hallucinated tool calls":    "Validate tool names; handle KeyError gracefully",
    "Infinite loops":             "max_steps limit; repeated-call detection",
    "Context overflow":           "Truncate tool outputs; summarise old turns",
    "Stale tool over-reliance":   "System prompt rules on when to use tools",
    "Partial action execution":   "Confirm before side-effects; transactional actions",
    "Prompt injection via tools": "Sanitise tool outputs before re-injecting to LLM",
}
for mode, fix in FAILURE_MODES.items():
    print(f"  ❌ {mode:<38} → Fix: {fix}")

=== Failure Mode 1: Hallucinated tool calls ===
[ERROR] Tool 'nonexistent_tool' does not exist. Available: ['get_weather', 'calculator', 'get_stock_price']
12*8 = 96

=== Failure Mode 2: Infinite loops ===
Loop detected: True → stop and return partial answer

=== Failure Mode 3: Context window overflow ===
Original length:  2000
Truncated length: 533

=== Failure mode checklist ===
  ❌ Hallucinated tool calls                → Fix: Validate tool names; handle KeyError gracefully
  ❌ Infinite loops                         → Fix: max_steps limit; repeated-call detection
  ❌ Context overflow                       → Fix: Truncate tool outputs; summarise old turns
  ❌ Stale tool over-reliance               → Fix: System prompt rules on when to use tools
  ❌ Partial action execution               → Fix: Confirm before side-effects; transactional actions
  ❌ Prompt injection via tools             → Fix: Sanitise tool outputs before re-injecting to LLM


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag